# The drone as a measuring instrument: timing, position and angles on a real flight

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/drone_hardware_en.ipynb)

**OpenSkyLab · notebook for the lab** [The drone as a measuring instrument](https://openskylab.space/edu_viz_drone_hardware.html) · [česká verze](drone_hardware_cs.ipynb)

The lab shows that a drone is a set of instruments: a GNSS receiver, a barometer, an inertial unit, a gimbal and a camera, each measuring in a slightly different place and at a slightly different moment. Here you test this on a real mapping flight over Bahna (DJI Mavic 2 Pro, no RTK). For each image you have the position and heights recorded by the drone, the body and camera orientation and the camera centre computed by photogrammetry (SfM). From their differences you find how fast the drone flew, how it tilts, how much the position record lags and how much the image is blurred.

**Goal:** determine the strips, speed and image spacing from image positions and times and find missing images, compare the body and camera tilt in strips flown in both directions, separate the position-record delay from the antenna lever arm, compute motion blur and compare the GNSS and barometric heights.

**Data:** a metadata table of 253 images of a flight over Bahna on 31 Oct 2025 (EXIF and DJI XMP) with SfM camera centres, in local coordinates (OpenSkyLab, CC BY 4.0).

**Tasks:** five. Write your answers in the *Your answer* cells below the tasks. Lines marked `# ◀ parameter` are meant to be changed; run the cell again with Shift + Enter.

**Time:** about 45 min. You need a Google account (Colab); nothing is installed on your computer. Run the cells in order from top to bottom.

## 1. Data and flight track

The table has one row per image. Positions are in a local plane in metres (east, north), heights above the take-off point. We split the mapping flight (`grid`) into strips by jumps in the north position and give each strip a direction.

**Required cell, run it first.** Loads the libraries. The code is collapsed; reveal it with **Show code**.

In [ ]:
# @title Setup (run first) {display-mode: "form"}
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

**Required cell, run it right after the setup.** Pasting the settings is optional: if you paste the settings copied from the lab into the `Z_VIZUALU` field, the cells below take the same parameter values and say so for each one. If you leave the field empty (`{}`), the default values are used. Run the cell in either case, because it defines the function `z_viz` that later cells use. It downloads nothing. The code is collapsed; reveal it with **Show code**.

In [ ]:
# @title Settings from the lab (run it; pasting is optional) {display-mode: "form"}
# @markdown In the lab, click the Notebook link. The settings are copied; paste them here in place of `{}` (Ctrl+V) and run the cell.
Z_VIZUALU = {}  # @param {type:"raw"}
STEM = "drone_hardware"


def z_viz(name, default, mapping=None):
    """Value of a parameter from the lab if the pasted code contains it; otherwise the default value."""
    z = (
        Z_VIZUALU
        if isinstance(Z_VIZUALU, dict) and Z_VIZUALU.get("osl") == STEM
        else {}
    )
    if name not in z:
        return default
    v = z[name]
    if mapping is not None:
        if v not in mapping:
            print(
                f"{name}: value {v!r} from the lab is not in this notebook, keeping {default!r}"
            )
            return default
        v = mapping[v]
    elif isinstance(default, bool):
        v = bool(v)
    elif isinstance(default, int):
        v = int(round(v))
    elif isinstance(default, float):
        v = float(v)
    print(f"{name} = {v!r}  (from the lab)")
    return v


if isinstance(Z_VIZUALU, dict) and Z_VIZUALU and Z_VIZUALU.get("osl") != STEM:
    print(
        f"The pasted code is from another lab ({Z_VIZUALU.get('osl')}), the notebook does not use it."
    )

> **💻 What the code does:** Downloads the metadata table of the flight over Bahna (one row per image) from the notebook repository, selects the mapping flight, splits it into strips by jumps in the north position and gives each strip a direction. It prints the numbers of images and strips, the flight duration and height and draws the track coloured by flight direction. The data is from OpenSkyLab under the CC BY 4.0 licence.

In [ ]:
URL = "https://raw.githubusercontent.com/OpenSkyLab-space/notebooks/main/data/drone_hardware_bahna_flight.csv"
d = pd.read_csv(URL)
# The mapping flight in strips.
g = d[d.set == "grid"].sort_values("t_s").reset_index(drop=True)
# a new strip starts when the north position jumps by more than 5 m (strips are about 25 m apart)
g["strip"] = (g.n_gnss.diff().abs() > 5).cumsum()
first, last = g.groupby("strip").e_gnss.first(), g.groupby("strip").e_gnss.last()
# +1 = eastwards, −1 = westwards.
g["dir"] = g.strip.map(np.sign(last - first)).astype(int)
print(
    f"images: {len(d)} (mapping flight {len(g)}, extra {(d.set == 'extra').sum()}), strips: {g.strip.nunique()}"
)
print(
    f"the mapping flight took {g.t_s.max() / 60:.1f} min, height above take-off {g.h_baro.median():.1f} m"
)
fig, ax = plt.subplots(figsize=(8, 5))
for s, q in g.groupby("strip"):
    ax.plot(
        q.e_gnss,
        q.n_gnss,
        ".-",
        ms=4,
        lw=0.6,
        c="tab:red" if q.dir.iloc[0] > 0 else "tab:blue",
    )
ax.plot([], [], "r.-", label="eastwards")
ax.plot([], [], "b.-", label="westwards")
ax.set_aspect("equal")
ax.set_xlabel("E (m)")
ax.set_ylabel("N (m)")
ax.legend()
ax.set_title("Flight track from the drone GNSS")
plt.show()

## 2. Speed, spacing and missing images

The drone takes images at a regular interval. The speed in each strip is the slope of a line fitted to the east position against time. EXIF time has whole seconds only, hence a line rather than differences of neighbouring images.

> **💻 What the code does:** For each strip it computes the speed as the slope of the east position against time and the median interval and spacing of the images, and shows them as a table. Then it finds missing image numbers and places within strips where the spacing is more than 1.5× the usual one.

In [ ]:
rows = []
for s, q in g.groupby("strip"):
    v = abs(np.polyfit(q.t_s, q.e_gnss, 1)[0])  # speed = slope of the line E(t)
    rows.append(
        {
            "strip": s,
            "direction": "E" if q.dir.iloc[0] > 0 else "W",
            "images": len(q),
            "v (m/s)": round(v, 2),
            "interval (s)": q.t_s.diff().median(),
            "spacing (m)": round(
                np.hypot(q.e_gnss.diff(), q.n_gnss.diff()).median(), 1
            ),
        }
    )
st = pd.DataFrame(rows)
display(st)
V_DATA = st["v (m/s)"].median()
print(f"median speed {V_DATA:.2f} m/s")
num = g.image.str.extract(r"_(\d{4})", expand=False).astype(int)
missing = sorted(set(range(num.min(), num.max() + 1)) - set(num))
print(f"missing image numbers: {missing}")
step = np.hypot(g.e_gnss.diff(), g.n_gnss.diff())
same = g.strip.diff() == 0
big = g[same & (step > 1.5 * step[same].median())]
print("within a strip the spacing exceeds 1.5 × median before images:", list(big.image))

> **✏️ Task 1.** What are the speed, interval and image spacing? Do eastward and westward strips differ? Which images are missing and can you tell from the spacing where in a strip there is a hole?

*Your answer:* 

## 3. Body and camera tilt

A multicopter flies forward by tilting, so that the rotor thrust gets a horizontal component. The gimbal is meant to cancel the body tilt so that the camera points straight down.

> **💻 What the code does:** Computes the mean and standard deviation of the drone body pitch, roll and heading and of the camera pitch and heading separately for eastward and westward strips. It plots the body and camera pitch and the body heading over time.

In [ ]:
att = (
    g.groupby("dir")[
        ["flight_pitch", "flight_roll", "flight_yaw", "gimbal_pitch", "gimbal_yaw"]
    ]
    .agg(["mean", "std"])
    .round(1)
)
att.index = att.index.map({1: "eastwards", -1: "westwards"})
display(att)
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(g.t_s, g.flight_pitch, ".", ms=3, label="body pitch (flight_pitch)")
ax[0].plot(
    g.t_s,
    g.gimbal_pitch + 90,
    ".",
    ms=3,
    label="camera pitch + 90° (gimbal_pitch + 90)",
)
ax[0].axhline(0, c="k", lw=0.5)
ax[0].set_ylabel("°")
ax[0].legend(fontsize=8)
ax[1].plot(g.t_s, g.flight_yaw, ".", ms=3, label="body heading (flight_yaw)")
ax[1].plot(
    g.t_s,
    g.e_gnss.diff().apply(np.sign) * 45 + 90,
    "-",
    lw=0.5,
    c="gray",
    label="flight direction (up E, down W)",
)
ax[1].set_ylabel("°")
ax[1].set_xlabel("time since the first image (s)")
ax[1].legend(fontsize=8)
plt.show()

> **✏️ Task 2.** Where does the drone's nose point in eastward and westward strips? What is the body pitch in both directions and why does it have opposite signs? Did the gimbal keep the camera pointing straight down? Why is the tilt not equally large eastwards and westwards?

*Your answer:* 

## 4. Position-record delay and lever arm

The SfM camera centre is determined from the images, the EXIF position from GNSS. If the position is recorded dt after the exposure, it is shifted by v · dt in the flight direction, so it has opposite signs in strips flown back and forth. An antenna offset from the camera (lever arm) gives a shift that does not change with flight direction, because the drone's nose always points the same way. Note: SfM was fitted to the GNSS positions by a similarity transformation, so any constant shift of the whole block vanished and cannot be determined from the data.

> **💻 What the code does:** Takes the speed, delay and lever arm from the lab (or default values), computes the difference between the GNSS position and the SfM camera centre and its mean in eastward and westward strips. From half the difference of the two means it estimates the shift v · dt and the delay dt, from their mean the constant offset, and compares them with the lab. It also computes the antenna shift from body tilt and draws histograms of the differences.

In [ ]:
SPEED = z_viz("speed", 8.0)  # ◀ parameter: flight speed in the lab (m/s)
DT = z_viz("dt", 0.24)  # ◀ parameter: position-record delay in the lab (s)
# Antenna ahead of the camera in the lab (m).
LEVER_X = z_viz("lever_x", 0.0)  # ◀ parameter
LEVER_Z = z_viz("lever_z", 0.2)  # ◀ parameter: antenna above the camera in the lab (m)
ok = g.e_sfm.notna()
g["dE"] = g.e_gnss - g.e_sfm  # GNSS minus SfM camera centre, east
g["dN"] = g.n_gnss - g.n_sfm
m = g[ok].groupby("dir")[["dE", "dN"]].agg(["mean", "std", "size"]).round(2)
m.index = m.index.map({1: "eastwards", -1: "westwards"})
display(m)
dE_e, dE_w = g[ok & (g.dir > 0)].dE.mean(), g[ok & (g.dir < 0)].dE.mean()
# model: dE = direction · v · dt + constant offset (lever arm; the drone nose always points east)
vdt = (dE_e - dE_w) / 2
const = (dE_e + dE_w) / 2
print(
    f"shift in the flight direction v·dt = {vdt:.2f} m  →  dt = {vdt / V_DATA:.3f} s at v = {V_DATA:.2f} m/s"
)
print(f"lab: v·dt = {SPEED * DT:.2f} m (v = {SPEED} m/s, dt = {DT} s)")
print(f"constant offset to the east: {const:+.2f} m")
print(
    f"lab: an antenna {LEVER_X:.2f} m ahead of the camera would give a constant offset {LEVER_X:+.2f} m to the east (the nose points east)"
)
tilt = LEVER_Z * np.sin(np.radians(g.groupby("dir").flight_pitch.mean().abs().mean()))
print(f"antenna shift from body tilt (lever_z · sin|pitch|): {tilt:.3f} m")
fig, ax = plt.subplots(figsize=(8, 4))
for s_, c, lab in ((1, "tab:red", "eastwards"), (-1, "tab:blue", "westwards")):
    ax.hist(g[ok & (g.dir == s_)].dE, bins=30, alpha=0.6, color=c, label=lab)
ax.axvline(0, c="k", lw=0.5)
ax.set_xlabel("E_GNSS − E_SfM (m)")
ax.legend()
plt.show()

> **✏️ Task 3.** What delay dt follows from the data and how does it agree with the lab? Why is the constant offset almost zero even if the antenna were ahead of the camera? How large a shift does the body tilt cause with the antenna 0.2 m above the camera, and can it be told apart from the delay?

*Your answer:* 

## 5. Motion blur and shutter

During the exposure the drone moves by v · t_exp. If that is more than about half a ground pixel (GSD), the image is visibly blurred. The L1D-20c camera has no mechanical shutter: the sensor is read out row by row (rolling shutter) and during readout the drone moves by v · t_read.

> **💻 What the code does:** Computes the GSD from the height, focal length and pixel size, the blur in centimetres and pixels from the strip speed and exposure time, and shows them by exposure time. For comparison it prints the blur for the lab's exposure time and the shift during sensor readout `T_READ`.

In [ ]:
# Focal length and pixel size of the Hasselblad L1D-20c (image EXIF: f = 10.26 mm, 28 mm
# equivalent, 5472 px).
F_MM, PIX_UM = 10.26, 2.41
T_EXP = z_viz("t_exp", 0.002)  # ◀ parameter: exposure time in the lab (s)
# Sensor readout time in the lab (s), 0 = mechanical shutter.
T_READ = z_viz("t_read", 0.03)  # ◀ parameter
g["gsd_cm"] = g.h_baro * PIX_UM * 1e-6 / (F_MM * 1e-3) * 100
g["v"] = g.strip.map(dict(zip(st.iloc[:, 0], st["v (m/s)"])))
g["blur_cm"] = g.v * g.exposure_s * 100
g["blur_px"] = g.blur_cm / g.gsd_cm
tab = (
    g.groupby("exposure_s")
    .agg(
        n=("image", "size"),
        gsd_cm=("gsd_cm", "median"),
        blur_cm=("blur_cm", "median"),
        blur_px=("blur_px", "median"),
    )
    .round(2)
)
tab.index = [f"1/{1 / e:.0f} s" for e in tab.index]
display(tab)
print(
    f"lab: t_exp = 1/{1 / T_EXP:.0f} s → blur {SPEED * T_EXP * 100:.1f} cm = {SPEED * T_EXP * 100 / g.gsd_cm.median():.2f} px"
)
print(
    f"shift during sensor readout (t_read = {T_READ} s): {V_DATA * T_READ * 100:.0f} cm = {V_DATA * T_READ * 100 / g.gsd_cm.median():.0f} px between the first and last row of the image"
)

> **✏️ Task 4.** Which exposure times did the camera use and how large is the blur in pixels? What is the longest time that would keep the blur below 0.5 px at this speed and height? What would you change in the flight plan if it were darker under clouds?

*Your answer:* 

## 6. GNSS and barometric height

The drone measures height with a barometer (pressure difference to the take-off point) and with the GNSS receiver. `h_gnss` is the absolute altitude from XMP shifted so that it can be compared with `h_baro`.

> **💻 What the code does:** Computes the difference between the GNSS and barometric heights, its mean, spread, range and trend over time, and the spread of the difference to the SfM height. It plots all three heights over time.

In [ ]:
g["dh"] = g.h_gnss - g.h_baro
print(
    f"h_gnss − h_baro: mean {g.dh.mean():+.2f} m, standard deviation {g.dh.std():.2f} m, range {g.dh.min():+.2f} … {g.dh.max():+.2f} m"
)
k = np.polyfit(g.t_s, g.dh, 1)[0] * 60
print(f"trend h_gnss − h_baro: {k:+.3f} m/min")
okh = g.h_sfm.notna()
print(
    f"h_sfm − h_baro: standard deviation {(g.h_sfm - g.h_baro)[okh].std():.2f} m;  h_sfm − h_gnss: {(g.h_sfm - g.h_gnss)[okh].std():.2f} m"
)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(g.t_s, g.h_baro, ".", ms=3, label="h_baro (barometer)")
ax.plot(g.t_s, g.h_gnss, ".", ms=3, label="h_gnss (GNSS)")
ax.plot(g.t_s[okh], g.h_sfm[okh], ".", ms=3, label="h_sfm (SfM)")
ax.set_xlabel("time since the first image (s)")
ax.set_ylabel("height above take-off (m)")
ax.legend()
plt.show()

> **✏️ Task 5.** Which sensor keeps the height more stable and why? Does the difference h_gnss − h_baro have a trend in time? What can shift the barometer during the flight?

*Your answer:* 

> **✏️ Extension (optional).** The extra images (`set == 'extra'`) have a different camera tilt and height. Which of them were not registered in SfM and why?

*Your answer:* 

---
### Data sources and references

- Flight data: OpenSkyLab (2026), metadata of the Bahna flight of 31 Oct 2025 derived from the images and the SfM reconstruction, licence CC BY 4.0, https://github.com/OpenSkyLab-space/notebooks/tree/main/data
- Pix4D: *How to correct for the rolling shutter effect* and *Time offset between GNSS and camera* (Pix4Dmapper documentation).
- Vautherin, J., Rutishauser, S., Schneider-Zapp, K., Choi, H. F., Chovancova, V., Glass, A., Strecha, C. (2016). Photogrammetric accuracy and modeling of rolling shutter cameras. *ISPRS Annals* III-3, 139–146.
- Groves, P. D. (2013). *Principles of GNSS, Inertial, and Multisensor Integrated Navigation Systems*, 2nd ed. Artech House (lever arm, sensor synchronisation).

### How to cite

OpenSkyLab (2026). The drone as a measuring instrument: timing, position and angles on a real flight [notebook, teaching resource], version 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/drone_hardware_en.ipynb. Licence CC BY 4.0.

Notebook text and code: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data are under their providers' licences (see above).

*This resource was developed by the OpenSkyLab team with the assistance of the generative AI Claude (Anthropic), used for drafting code and text. The authors are responsible for the content.*